# 00 - Data Cleaning and Validation

This notebook is the **first stage** of the production workflow.

Goals:
- Connect to production-like source tables.
- Validate table-level quality.
- Standardize types and business-critical fields.
- Save clean snapshots for downstream notebooks.


## Notebook Rules

- Run top-to-bottom.
- Do not skip data contract checks.
- If assertions fail, fix data upstream before modeling.


In [ ]:
# Cell 0: Setup
from __future__ import annotations

from pathlib import Path
import json
import os

import numpy as np
import pandas as pd
from sqlalchemy import inspect, text

from ml.pipelines.churn_common import create_db_engine

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_rows', 200)

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent.parent
elif ROOT.name != 'Sliceiq' and (ROOT / 'ml').exists():
    pass

CLEAN_DIR = ROOT / 'ml' / 'data' / 'clean'
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

engine = create_db_engine(os.getenv('DATABASE_URL'))
inspector = inspect(engine)
print('Connected DB. Tables:', sorted(inspector.get_table_names()))


In [ ]:
# Cell 1: Resolve schema differences
orders_cols = {c['name'] for c in inspector.get_columns('orders')}
amount_col = 'total_amount' if 'total_amount' in orders_cols else 'total_price'

print('Using amount column:', amount_col)
print('orders columns:', sorted(orders_cols))


In [ ]:
# Cell 2: Load source tables
users = pd.read_sql(text('SELECT * FROM users'), engine)
orders = pd.read_sql(text('SELECT * FROM orders'), engine)

order_items = pd.DataFrame()
reviews = pd.DataFrame()
products = pd.DataFrame()

if 'order_items' in inspector.get_table_names():
    order_items = pd.read_sql(text('SELECT * FROM order_items'), engine)
if 'reviews' in inspector.get_table_names():
    reviews = pd.read_sql(text('SELECT * FROM reviews'), engine)
if 'products' in inspector.get_table_names():
    products = pd.read_sql(text('SELECT * FROM products'), engine)

print('users:', users.shape)
print('orders:', orders.shape)
print('order_items:', order_items.shape)
print('reviews:', reviews.shape)
print('products:', products.shape)


In [ ]:
# Cell 3: Quick previews
for name, df in [
    ('users', users),
    ('orders', orders),
    ('order_items', order_items),
    ('reviews', reviews),
    ('products', products),
]:
    if df.empty:
        continue
    print(f'\n{name} head:')
    display(df.head(3))


In [ ]:
# Cell 4: Missing values and duplicates
quality_rows = []
for name, df in [
    ('users', users),
    ('orders', orders),
    ('order_items', order_items),
    ('reviews', reviews),
    ('products', products),
]:
    if df.empty:
        continue
    quality_rows.append(
        {
            'table': name,
            'rows': len(df),
            'columns': df.shape[1],
            'missing_cells': int(df.isna().sum().sum()),
            'duplicate_rows': int(df.duplicated().sum()),
        }
    )

quality_df = pd.DataFrame(quality_rows).sort_values('table')
display(quality_df)


In [ ]:
# Cell 5: Type normalization
for df, ts_col in [
    (users, 'created_at'),
    (orders, 'created_at'),
    (reviews, 'created_at'),
    (products, 'created_at'),
]:
    if not df.empty and ts_col in df.columns:
        df[ts_col] = pd.to_datetime(df[ts_col], utc=True, errors='coerce')

if amount_col in orders.columns:
    orders[amount_col] = pd.to_numeric(orders[amount_col], errors='coerce')

if 'quantity' in order_items.columns:
    order_items['quantity'] = pd.to_numeric(order_items['quantity'], errors='coerce')
if 'unit_price' in order_items.columns:
    order_items['unit_price'] = pd.to_numeric(order_items['unit_price'], errors='coerce')
if 'rating' in reviews.columns:
    reviews['rating'] = pd.to_numeric(reviews['rating'], errors='coerce')


In [ ]:
# Cell 6: Data contract assertions
assert not users.empty, 'users table is empty'
assert not orders.empty, 'orders table is empty'

assert 'id' in users.columns, 'users.id missing'
assert 'id' in orders.columns, 'orders.id missing'
assert users['id'].astype(str).is_unique, 'users.id must be unique'
assert orders['id'].astype(str).is_unique, 'orders.id must be unique'

assert 'user_id' in orders.columns, 'orders.user_id missing'
assert orders['user_id'].notna().all(), 'orders.user_id has nulls'
assert orders[amount_col].notna().all(), f'orders.{amount_col} has nulls'

if not reviews.empty and 'rating' in reviews.columns:
    bad_ratings = reviews['rating'].dropna().loc[(reviews['rating'] < 1) | (reviews['rating'] > 5)]
    assert bad_ratings.empty, 'reviews.rating out of [1,5] range'

print('All core data contract checks passed.')


In [ ]:
# Cell 7: Final cleaned DataFrames
users_clean = users.drop_duplicates().copy()
orders_clean = orders.drop_duplicates().copy()
order_items_clean = order_items.drop_duplicates().copy()
reviews_clean = reviews.drop_duplicates().copy()
products_clean = products.drop_duplicates().copy()

for col in ['id', 'user_id', 'order_id', 'product_id']:
    if col in users_clean.columns:
        users_clean[col] = users_clean[col].astype(str)
    if col in orders_clean.columns:
        orders_clean[col] = orders_clean[col].astype(str)
    if col in order_items_clean.columns:
        order_items_clean[col] = order_items_clean[col].astype(str)
    if col in reviews_clean.columns:
        reviews_clean[col] = reviews_clean[col].astype(str)
    if col in products_clean.columns:
        products_clean[col] = products_clean[col].astype(str)


In [ ]:
# Cell 8: Save artifacts for downstream notebooks
users_clean.to_parquet(CLEAN_DIR / 'users_clean.parquet', index=False)
orders_clean.to_parquet(CLEAN_DIR / 'orders_clean.parquet', index=False)
if not order_items_clean.empty:
    order_items_clean.to_parquet(CLEAN_DIR / 'order_items_clean.parquet', index=False)
if not reviews_clean.empty:
    reviews_clean.to_parquet(CLEAN_DIR / 'reviews_clean.parquet', index=False)
if not products_clean.empty:
    products_clean.to_parquet(CLEAN_DIR / 'products_clean.parquet', index=False)

quality_payload = {
    'saved_at_utc': pd.Timestamp.utcnow().isoformat(),
    'amount_column': amount_col,
    'row_counts': {
        'users_clean': int(len(users_clean)),
        'orders_clean': int(len(orders_clean)),
        'order_items_clean': int(len(order_items_clean)),
        'reviews_clean': int(len(reviews_clean)),
        'products_clean': int(len(products_clean)),
    },
}

quality_report_path = CLEAN_DIR / 'cleaning_quality_report.json'
quality_report_path.write_text(json.dumps(quality_payload, indent=2), encoding='utf-8')
print('Saved clean artifacts to:', CLEAN_DIR)
print('Saved quality report:', quality_report_path)


## Next Notebook

Proceed to `01_advanced_eda.ipynb`.
